# T29 — Exam Countdown Planner

## Project Overview

The Exam Countdown Planner is an agent that creates and manages a
day-by-day study plan based on the user's exam date and study topics.

The agent uses Microsoft Foundry with `gpt-5-mini` as its language model
and uses Python functions as tools.

### Main capabilities

- Save the exam date
- Allocate study topics across the available days
- Remember the current study plan
- Detect when a study day was missed
- Adjust the existing plan using a catch-up shuffle

### Agent Tools

1. `set_exam(date)`
2. `allocate_topics(topics)`
3. `catch_up(missed_date)`

## Agent Architecture

The system follows an agentic workflow:

User
  ↓
Exam Countdown Agent
  ↓
gpt-5-mini
  ↓
Selects an appropriate tool
  ↓
Python Tool
  ↓
Tool Result
  ↓
Agent observes the result
  ↓
Next action or final response

### Components

- **LLM:** `gpt-5-mini`
- **Agent Framework:** Microsoft Agent Framework
- **Model Provider:** Microsoft Foundry
- **Tools:** `set_exam`, `allocate_topics`, `catch_up`
- **Memory:** Python `memory` dictionary containing the exam date,
  topics, and schedule

The agent is different from a simple chatbot because it can decide
which tool to call, execute that tool, observe its result, and use
the resulting state to perform subsequent actions.

In [1]:
import sys

print(sys.executable)
print(sys.version)

C:\Users\hp\miniconda3\envs\cse476\python.exe
3.12.13 | packaged by conda-forge | (main, Aug 11 2026, 10:18:27) [MSC v.1944 64 bit (AMD64)]


In [14]:
from datetime import datetime, timedelta
from typing import Annotated

from pydantic import Field
from azure.identity import AzureCliCredential

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient

## Microsoft Foundry Connection

The agent uses Azure CLI authentication through `AzureCliCredential`.
No API key is stored directly in the notebook.

In [ ]:
credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint="ENTER YOUR ENDPOINT HERE",
    model="gpt-5-mini",
    credential=credential
)

print("Foundry client ready!")

Foundry client ready!


## Agent Memory

The agent maintains a simple in-memory state for the current study plan.

The memory stores:

- `exam_date` — the user's exam date
- `topics` — the study topics
- `schedule` — the generated day-by-day plan

This is session-level memory. It remains available while the Python
kernel is running.

In [17]:
memory = {
    "exam_date": None,
    "topics": [],
    "schedule": []
}

print("Memory initialized:")
print(memory)

Memory initialized:
{'exam_date': None, 'topics': [], 'schedule': []}


## Tool 1 — set_exam()

This tool saves the exam date in the agent's memory.

The agent can decide to call this tool when the user provides an
exam date.

In [18]:
def set_exam(date):
    print(f"\n TOOL CALL: set_exam({date})")

    memory["exam_date"] = date

    result = f"Exam date saved: {date}"

    print(f" TOOL RESULT: {result}")

    return result

## Tool 2 — allocate_topics()

This tool creates a day-by-day study schedule.

It calculates the number of available study days from today's date
until the exam date and distributes the supplied topics across
those days.

The exam date is reserved for final revision.

In [19]:
def allocate_topics(
    topics: Annotated[
        list[str],
        Field(
            description=(
                "A list of study topic names, for example "
                "['Java', 'OOP', 'SQL', 'Spark']."
            )
        )
    ]
):
    print(f"\n TOOL CALL: allocate_topics({topics})")

    if not memory["exam_date"]:
        result = "Please set the exam date first."
        print(f" TOOL RESULT: {result}")
        return result

    if not topics:
        result = "No topics were provided."
        print(f" TOOL RESULT: {result}")
        return result

    exam_date = datetime.strptime(
        memory["exam_date"],
        "%Y-%m-%d"
    ).date()

    today = datetime.now().date()

    days_left = (exam_date - today).days

    if days_left <= 0:
        result = "The exam date has already passed."
        print(f" TOOL RESULT: {result}")
        return result

    study_days = days_left
    topic_count = len(topics)

    base_days = study_days // topic_count
    extra_days = study_days % topic_count

    schedule = []
    current_date = today

    for i, topic in enumerate(topics):

        topic_days = base_days

        if i < extra_days:
            topic_days += 1

        for _ in range(topic_days):

            schedule.append({
                "date": current_date.isoformat(),
                "topic": topic
            })

            current_date += timedelta(days=1)

    schedule.append({
        "date": exam_date.isoformat(),
        "topic": "Final revision"
    })

    memory["topics"] = topics
    memory["schedule"] = schedule

    print(f" TOOL RESULT: {schedule}")

    return schedule

## Tool 3 — catch_up()

This tool handles a missed study day.

When the user reports a missed date, the tool:

1. Finds the missed study task.
2. Removes the missed task from that date.
3. Finds the next available study day.
4. Merges the missed work into that day's topic.
5. Updates the stored schedule.

In [20]:
def catch_up(missed_date):
    print(f"\n TOOL CALL: catch_up({missed_date})")

    if not memory["schedule"]:
        result = "There is no study schedule to adjust."
        print(f" TOOL RESULT: {result}")
        return result

    missed_date_obj = datetime.strptime(
        missed_date,
        "%Y-%m-%d"
    ).date()

    missed_date_str = missed_date_obj.isoformat()

    missed_items = [
        item
        for item in memory["schedule"]
        if item["date"] == missed_date_str
        and item["topic"] != "Final revision"
    ]

    if not missed_items:
        result = f"No study task was scheduled for {missed_date}."
        print(f" TOOL RESULT: {result}")
        return result

    memory["schedule"] = [
        item
        for item in memory["schedule"]
        if item["date"] != missed_date_str
    ]

    next_days = [
        item["date"]
        for item in memory["schedule"]
        if item["date"] > missed_date_str
        and item["topic"] != "Final revision"
    ]

    if not next_days:
        result = "There are no remaining study days for catch-up."
        print(f" TOOL RESULT: {result}")
        return result

    catchup_date = min(next_days)

    for missed_item in missed_items:

        existing = next(
            (
                item
                for item in memory["schedule"]
                if item["date"] == catchup_date
                and item["topic"] == missed_item["topic"]
            ),
            None
        )

        if existing:
            existing["topic"] += " + catch-up"
        else:
            memory["schedule"].append({
                "date": catchup_date,
                "topic": missed_item["topic"] + " (catch-up)"
            })

    memory["schedule"].sort(
        key=lambda x: x["date"]
    )

    result = memory["schedule"]

    print(f" TOOL RESULT: {result}")

    return result

## Agent Definition

The agent uses `gpt-5-mini` as its reasoning model and has access to
three Python tools.

The agent decides which tool to use based on the user's request.

- `set_exam()` saves or updates the exam date.
- `allocate_topics()` creates the study schedule.
- `catch_up()` modifies the schedule when a study day is missed.

The agent can use the result of one tool to determine its next action.

In [21]:
agent = Agent(
    client=client,
    name="ExamCountdownPlanner",
    instructions="""
You are an Exam Countdown Planner.

Use the available tools to manage the user's exam plan.

Use set_exam when the user provides or changes an exam date.

Use allocate_topics when the user provides study topics and needs
a day-by-day study plan.

Use catch_up when the user reports a missed study day.

When a request contains both an exam date and study topics,
call set_exam first and then allocate_topics.

After a tool returns a result, use that result to decide what
to do next.

Do not invent a different schedule after a planning tool returns
a schedule. Present the schedule returned by the tool.
""",
    tools=[
        set_exam,
        allocate_topics,
        catch_up
    ]
)

print("Exam Countdown Planner agent created successfully!")

Exam Countdown Planner agent created successfully!


## Demo 1 — Create an Exam Study Plan

The user provides an exam date and a set of study topics.

The agent should:

1. Save the exam date using `set_exam()`.
2. Allocate the topics using `allocate_topics()`.
3. Present the schedule returned by the tools.

In [22]:
response = await agent.run(
    "My exam is on 2026-09-05. "
    "I need to study Java, OOP, SQL, and Spark."
)

print("\nFINAL AGENT RESPONSE:")
print(response.text)


 TOOL CALL: set_exam(2026-09-05)
 TOOL RESULT: Exam date saved: 2026-09-05

 TOOL CALL: allocate_topics(['Java', 'OOP', 'SQL', 'Spark'])
 TOOL RESULT: [{'date': '2026-08-26', 'topic': 'Java'}, {'date': '2026-08-27', 'topic': 'Java'}, {'date': '2026-08-28', 'topic': 'Java'}, {'date': '2026-08-29', 'topic': 'OOP'}, {'date': '2026-08-30', 'topic': 'OOP'}, {'date': '2026-08-31', 'topic': 'OOP'}, {'date': '2026-09-01', 'topic': 'SQL'}, {'date': '2026-09-02', 'topic': 'SQL'}, {'date': '2026-09-03', 'topic': 'Spark'}, {'date': '2026-09-04', 'topic': 'Spark'}, {'date': '2026-09-05', 'topic': 'Final revision'}]

FINAL AGENT RESPONSE:
Exam date saved: 2026-09-05.

Here is your day-by-day study schedule:

- 2026-08-26: Java
- 2026-08-27: Java
- 2026-08-28: Java
- 2026-08-29: OOP
- 2026-08-30: OOP
- 2026-08-31: OOP
- 2026-09-01: SQL
- 2026-09-02: SQL
- 2026-09-03: Spark
- 2026-09-04: Spark
- 2026-09-05: Final revision

If you want this adjusted (different topic lengths, more review days, or remin

## Memory After Initial Planning

The generated exam date, topics, and schedule are stored in the
agent's session memory.

In [23]:
print("CURRENT MEMORY:")
print(memory)

CURRENT MEMORY:
{'exam_date': '2026-09-05', 'topics': ['Java', 'OOP', 'SQL', 'Spark'], 'schedule': [{'date': '2026-08-26', 'topic': 'Java'}, {'date': '2026-08-27', 'topic': 'Java'}, {'date': '2026-08-28', 'topic': 'Java'}, {'date': '2026-08-29', 'topic': 'OOP'}, {'date': '2026-08-30', 'topic': 'OOP'}, {'date': '2026-08-31', 'topic': 'OOP'}, {'date': '2026-09-01', 'topic': 'SQL'}, {'date': '2026-09-02', 'topic': 'SQL'}, {'date': '2026-09-03', 'topic': 'Spark'}, {'date': '2026-09-04', 'topic': 'Spark'}, {'date': '2026-09-05', 'topic': 'Final revision'}]}


## Demo 2 — Catch-Up After a Missed Study Day

The user reports that a scheduled study day was missed.

The agent should recognize that the existing study plan needs to
be modified and independently call `catch_up()`.

The existing memory from Demo 1 is intentionally preserved.

In [24]:
response = await agent.run(
    "I missed my Java study session on 2026-08-27. "
    "Please adjust my existing study plan."
)

print("\nFINAL AGENT RESPONSE:")
print(response.text)


 TOOL CALL: catch_up(2026-08-27)
 TOOL RESULT: [{'date': '2026-08-26', 'topic': 'Java'}, {'date': '2026-08-28', 'topic': 'Java + catch-up'}, {'date': '2026-08-29', 'topic': 'OOP'}, {'date': '2026-08-30', 'topic': 'OOP'}, {'date': '2026-08-31', 'topic': 'OOP'}, {'date': '2026-09-01', 'topic': 'SQL'}, {'date': '2026-09-02', 'topic': 'SQL'}, {'date': '2026-09-03', 'topic': 'Spark'}, {'date': '2026-09-04', 'topic': 'Spark'}, {'date': '2026-09-05', 'topic': 'Final revision'}]

FINAL AGENT RESPONSE:
Got it — I’ve updated your plan to account for the missed Java session on 2026-08-27. Here’s the adjusted day-by-day schedule returned by the planner:

- 2026-08-26: Java  
- 2026-08-28: Java + catch-up  
- 2026-08-29: OOP  
- 2026-08-30: OOP  
- 2026-08-31: OOP  
- 2026-09-01: SQL  
- 2026-09-02: SQL  
- 2026-09-03: Spark  
- 2026-09-04: Spark  
- 2026-09-05: Final revision

Quick tips for the catch-up day (2026-08-28):
- Start with 20–30 minutes reviewing what you missed (notes, key concepts).

## Final Memory State

After the catch-up operation, the agent's memory contains the
original exam information and the updated study schedule.

In [25]:
print("FINAL MEMORY:")
print(memory)

FINAL MEMORY:
{'exam_date': '2026-09-05', 'topics': ['Java', 'OOP', 'SQL', 'Spark'], 'schedule': [{'date': '2026-08-26', 'topic': 'Java'}, {'date': '2026-08-28', 'topic': 'Java + catch-up'}, {'date': '2026-08-29', 'topic': 'OOP'}, {'date': '2026-08-30', 'topic': 'OOP'}, {'date': '2026-08-31', 'topic': 'OOP'}, {'date': '2026-09-01', 'topic': 'SQL'}, {'date': '2026-09-02', 'topic': 'SQL'}, {'date': '2026-09-03', 'topic': 'Spark'}, {'date': '2026-09-04', 'topic': 'Spark'}, {'date': '2026-09-05', 'topic': 'Final revision'}]}


## Why This Is an Agent

A traditional chatbot primarily follows:

User → LLM → Response

This project uses an agentic workflow:

User
  ↓
Agent / LLM
  ↓
Selects a tool
  ↓
Tool executes an action
  ↓
Tool returns a result
  ↓
Agent observes the result
  ↓
Agent performs another action or produces a response

For example, during initial planning the agent calls:

`set_exam()` → `allocate_topics()`

When the user reports a missed study day, the agent independently
selects:

`catch_up()`

The agent therefore does more than generate text. It uses tools to
modify application state and adapts its actions based on tool
results.

## Conclusion

The Exam Countdown Planner demonstrates an agent built with
Microsoft Foundry, `gpt-5-mini`, and Python function tools.

The agent can:

- Save an exam date
- Allocate topics across the remaining study days
- Maintain session-level memory
- Detect a missed study day through user input
- Modify the existing schedule using a catch-up tool
- Select and execute tools based on the user's goal

The observable tool-call traces demonstrate the agent's
multi-step interaction with its tools.